# GR-TFiLM diffssl LSTM — Local Eval

Evaluates models trained by [`train_lstm_output_transformer_tfilm.ipynb`](train_lstm_output_transformer_tfilm.ipynb)
(`GRTFiLMDiffSSLLSTM`): the diffssl `LSTM32TVC` core (raw input, `tvcond` on `|x|` +
4 static knobs) with the exported GR curve added as a **temporal FiLM** (γ/β):

```
raw dry ─┐  tvcond (TVFiLMCond): pool(|x|)⊕knobs → cond_seq[16]
         └── cat(x, cond_seq) → main LSTM(17→32) → hidden
                                        │
              GR curve ─► GR-TFiLM: pool(GR) → blockLSTM → γ,β → modulate hidden
                                        │  Linear(32→1) → tanh → wet
```

- **Input**: raw dry (the model learns the gain itself — *no* amplitude matching)
- **Static knobs**: `TVFiLMCond` (SOTA `tvcond`), from the setting name via `normalize_setting_params`
- **GR conditioning**: the sample-aligned `gr_db` curve drives the GR-TFiLM
- **Target**: wet audio
- **Stateful inference**: whole-song streaming with `reset_states()` at each pair
  start and internal state (main LSTM + `tvcond` + GR-TFiLM) carried across chunks
- **Metrics**: the canonical 9-column engine ported from
  [`02b_sota_training/eval_lstm_diffssl_tvc.ipynb`](../02b_sota_training/eval_lstm_diffssl_tvc.ipynb) —
  **GR MAE (dB)**, **MR-STE**, **MR-STFT** (auraloss `MultiResolutionSTFTLoss`),
  **ESR (A-wt)** (auraloss ESR on A-weighted signals), **MAE**, **MSE**, **EDC**,
  **M_NRMSE**, **M_SF**. MR-STFT / ESR are verified against
  `nablafx.evaluation.get_function(...)` to <1e-3 / <1e-4; **GR MAE (dB)** is recovered
  from the waveforms (`GR = RMS_dB(out) − RMS_dB(dry)`, same 1024-sample causal RMS
  window as the exported curves) so wet-audio and GR-bins models stay directly comparable.

> Requirements: `uv sync` at repo root, plus a local `./nablafx/` clone (the
> `tvcond` conditioner is nablafx's `TVFiLMCond`). Run after the Colab run dir has
> synced to Drive.

In [1]:
# ── 0. Imports and defaults ─────────────────────────────────────────

import gc, glob, json, os, sys, types, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings("ignore", category=UserWarning)

# model_tfilm imports nablafx's TVFiLMCond. Stub the `rational` /
# `frechet_audio_distance` import-chain deps so `from nablafx...` doesn't drag in
# broken wheels — identical to the train notebook's cell 0.
_rational = types.ModuleType("rational")
_rational.torch = types.ModuleType("rational.torch")
_rational.torch.Rational = type("Rational", (), {})
sys.modules.setdefault("rational", _rational)
sys.modules.setdefault("rational.torch", _rational.torch)
_fad = types.ModuleType("frechet_audio_distance")
_fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules.setdefault("frechet_audio_distance", _fad)

REPO_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file()),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(REPO_ROOT))                       # `src`
sys.path.insert(0, str(REPO_ROOT / "nablafx"))           # raw ./nablafx/ clone (TVFiLMCond)
sys.path.insert(0, str(REPO_ROOT / "06_output"))
sys.path.insert(0, str(REPO_ROOT / "03_initial_GR_pred"))  # eval_helpers

from amplitude_match import GR_DB_MIN, GR_DB_MAX
from model_tfilm import GRTFiLMDiffSSLLSTM
from splits import SplitManifest, normalize_setting_params
from system_tfilm import esr_metric
from eval_helpers import (
    _checkpoint_sort_key, _find_hparams_json, _pair_num_frames,
    _read_dry_wet_segment, _weighted_average_metric_rows, latest_best_checkpoint,
    list_runs, plot_loss_curves, wet_dir_for_setting,
)
from src.dsp_torch import RMS_WINDOW, gain_reduction_db

DATA_ROOT = "/Volumes/Saola's Drive/AllCode/thesis/data/Diff-SSL-G-Comp"
RUNS_DIR = "/Volumes/Saola's Drive/AllCode/thesis/data/diffssl_gr_tfilm_runs"
DEVICE = "cpu"

print(f"torch {torch.__version__}  |  device: {DEVICE}")
print(f"GR clamp: [{GR_DB_MIN}, {GR_DB_MAX}] dB  |  RMS window: {RMS_WINDOW} samples")

/Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.10.0  |  device: cpu
GR clamp: [-30.0, 5.0] dB  |  RMS window: 1024 samples


In [2]:
# ── 1. List available GR-TFiLM diffssl LSTM runs ─────────────────────

MODEL_TYPE = "GRTFiLMDiffSSLLSTM"   # hparams["model_type"]

runs_df = list_runs(RUNS_DIR)
assert not runs_df.empty, f"No runs in {RUNS_DIR}"
available_models_df = runs_df[runs_df["type"] == MODEL_TYPE].copy()
if available_models_df.empty:
    # fall back to folder-name tag if hparams["model_type"] is missing/older
    available_models_df = runs_df[runs_df["run"].str.contains("gr_tfilm", case=False, na=False)].copy()
assert not available_models_df.empty, (
    f"No '{MODEL_TYPE}' runs in {RUNS_DIR} — train with train_lstm_output_transformer_tfilm.ipynb first."
)

display_cols = ["run", "type", "epoch", "best_val", "n_ckpts", "eval_ckpt", "modified"]
print(f"Available GR-TFiLM runs: {len(available_models_df)}")
display(available_models_df[display_cols])


Available GR-TFiLM runs: 6


,run,type,epoch,best_val,n_ckpts,eval_ckpt,modified
0,gr_tfilm_20260701_162930_diffssl_lstm32_tvc_gr...,GRTFiLMDiffSSLLSTM,7.0,0.112904,4,best-005-73080.ckpt,2026-07-01 21:45:25.561416388
1,gr_tfilm_20260701_165533_diffssl_lstm32_tvc_gr...,GRTFiLMDiffSSLLSTM,100.0,0.167996,4,best-097-59388.ckpt,2026-07-01 21:45:25.884961605
2,gr_tfilm_20260701_193743_diffssl_lstm32_tvc_gr...,GRTFiLMDiffSSLLSTM,99.0,0.132679,4,best-099-303000.ckpt,2026-07-01 21:45:26.219090700
3,gr_tfilm_20260701_194007_diffssl_lstm32_tvc_gr...,GRTFiLMDiffSSLLSTM,NaN,NaN,0,None,2026-07-01 21:45:26.607988596
4,gr_tfilm_20260701_194303_diffssl_lstm32_tvc_gr...,GRTFiLMDiffSSLLSTM,NaN,NaN,0,None,2026-07-01 21:45:27.073838949
5,gr_tfilm_20260701_195700_diffssl_lstm32_tvc_gr...,GRTFiLMDiffSSLLSTM,31.0,0.120756,4,best-023-292320.ckpt,2026-07-01 21:45:27.249441147


In [3]:
# ── 2. Select run, checkpoint and split manifest ─────────────────────

# Leave as None to auto-pick the most recently modified matching run.
SELECTED_RUN_NAME: str | None = None

if not SELECTED_RUN_NAME:
    selected_run = available_models_df.iloc[-1]   # list_runs sorts by mtime asc
else:
    matches = available_models_df[available_models_df["run"] == SELECTED_RUN_NAME]
    assert len(matches) == 1, f"Run not found: {SELECTED_RUN_NAME}"
    selected_run = matches.iloc[0]

RUN_NAME = str(selected_run["run"])
RUN_DIR = os.path.join(RUNS_DIR, RUN_NAME)
CKPT_DIR = os.path.join(RUN_DIR, "checkpoints")
HPARAMS_PATH = os.path.join(RUN_DIR, "hparams.json")
EVAL_CKPT_PATH = latest_best_checkpoint(RUN_DIR)

with open(HPARAMS_PATH) as f:
    HP = json.load(f)
SPLIT = SplitManifest.load(os.path.join(RUN_DIR, "split_manifest.json"))


def _pairs_from_keys(keys) -> list[tuple[str, str]]:
    return [tuple(k.split("::")) for k in sorted(keys)]   # (song, setting)


VAL_PAIRS = _pairs_from_keys(SPLIT.val_pair_keys)
TEST_PAIRS = _pairs_from_keys(SPLIT.test_pair_keys)

print(f"Selected run   : {RUN_NAME}")
print(f"Eval checkpoint: {os.path.basename(EVAL_CKPT_PATH)}")
if pd.notna(selected_run.get("best_val")):
    print(f"Best val loss  : {selected_run['best_val']:.6f}")
print(f"Val pairs      : {len(VAL_PAIRS)}  ({SPLIT.val_songs} x all settings)")
print(f"Test pairs     : {len(TEST_PAIRS)}  ({SPLIT.test_songs} x {SPLIT.test_settings})")


Selected run   : gr_tfilm_20260701_195700_diffssl_lstm32_tvc_gr_tfilm
Eval checkpoint: best-023-292320.ckpt
Best val loss  : 0.120756
Val pairs      : 10  (['Ecstasy'] x all settings)
Test pairs     : 4  (['Air', 'AncoraQui'] x ['threshold_-12_attack_10_release_0.4_ratio_10', 'threshold_-12_attack_1_release_0.1_ratio_2'])


In [4]:
# ── 3. Inspect checkpoints ──────────────────────────────────────────

print(f"Selected run   : {RUN_NAME}")
print(f"Eval checkpoint: {os.path.basename(EVAL_CKPT_PATH)}")
print("Checkpoints    :")
for c in sorted(glob.glob(os.path.join(CKPT_DIR, "*.ckpt")), key=_checkpoint_sort_key):
    marker = " <-- eval" if os.path.abspath(c) == os.path.abspath(EVAL_CKPT_PATH) else ""
    sz = os.path.getsize(c) / 1e6
    print(f"  {os.path.basename(c):<36}  {sz:6.2f} MB{marker}")


Selected run   : gr_tfilm_20260701_195700_diffssl_lstm32_tvc_gr_tfilm
Eval checkpoint: best-023-292320.ckpt
Checkpoints    :
  last.ckpt                               0.32 MB
  best-020-255780.ckpt                    0.32 MB
  best-021-267960.ckpt                    0.32 MB
  best-023-292320.ckpt                    0.32 MB <-- eval


In [5]:
# ── 4. Loss curves (total + time / freq components) ─────────────────

_ = plot_loss_curves(RUN_DIR)

csv_path = Path(RUN_DIR) / "csv" / "metrics.csv"
if csv_path.exists():
    _df = pd.read_csv(csv_path)
    comp_cols = [
        ("loss/val_td", "L1 (val)"),
        ("loss/val_fd", "MR-STFT (val)"),
        ("esr/val", "ESR (val)"),
        ("mae/val", "MAE (val)"),
    ]
    if any(c in _df.columns for c, _ in comp_cols):
        fig, ax = plt.subplots(figsize=(8, 4))
        for col, label in comp_cols:
            if col in _df.columns:
                rows = _df.dropna(subset=[col])[["epoch", col]]
                if len(rows):
                    ax.plot(rows["epoch"], rows[col], label=label, lw=1.5)
        ax.set_xlabel("epoch")
        ax.set_ylabel("metric")
        ax.set_yscale("log")
        ax.grid(alpha=0.3, which="both")
        ax.legend()
        ax.set_title(f"{RUN_NAME} — waveform loss components (val)")
        plt.tight_layout()
        plt.show()


<Figure size 600x400 with 1 Axes>

<Figure size 800x400 with 1 Axes>

In [6]:
# ── 5. Checkpoint loading (GRTFiLMDiffSSLLSTM) ───────────────────────

def load_model_from_ckpt(ckpt_path, hparams_path=None):
    # Build from hparams.json and load weights (strip Lightning 'model.' prefix).
    hparams_path = hparams_path or _find_hparams_json(ckpt_path)
    with open(hparams_path) as f:
        hp = json.load(f)
    assert hp.get("model_type") == MODEL_TYPE, hp.get("model_type")
    m = hp["model"]
    model = GRTFiLMDiffSSLLSTM(
        num_controls=int(m["num_controls"]),
        hidden_size=int(m["hidden_size"]),
        num_layers=int(m["num_layers"]),
        tvcond_dim=int(m.get("tvcond_dim", 16)),
        cond_block_size=int(m.get("cond_block_size", 128)),
        cond_num_layers=int(m.get("cond_num_layers", 1)),
        gr_tfilm_block_size=int(m.get("gr_tfilm_block_size", 128)),
        gr_tfilm_num_layers=int(m.get("gr_tfilm_num_layers", 1)),
    )
    sd = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)["state_dict"]
    model.load_state_dict({k[len("model."):]: v for k, v in sd.items() if k.startswith("model.")})
    model.to(DEVICE).eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(
        f"Loaded {n_params:,}-param GRTFiLMDiffSSLLSTM  "
        f"(hidden={model.hidden_size}, controls={model.num_controls}, "
        f"cond_block={model.cond_block_size})"
    )
    return model, hp


MODEL, HP = load_model_from_ckpt(EVAL_CKPT_PATH, HPARAMS_PATH)
SR = int(HP["sample_rate"])
SAMPLE_LENGTH = int(HP.get("sample_length", 132300))
# GR derived from waveforms uses the same 1024-sample causal RMS window the
# exported gr_curves were built with (matches eval_lstm_condlstm_gr_bins.ipynb).
RMS_WIN = int(HP.get("rms_window", RMS_WINDOW))

Loaded 25,185-param GRTFiLMDiffSSLLSTM  (hidden=32, controls=4, cond_block=128)


In [7]:
# ── 6. Prediction & visualisation (stateful, raw input + GR-TFiLM) ───
# The model takes the RAW dry signal, the 4 static knobs (from the setting name),
# and the sample-aligned GR curve. A `pre_roll_sec` context is fed first (after
# reset_states) to warm all internal LSTM states, then the segment is cropped out.
#
# Besides the waveform, we recover a gain-reduction curve from the prediction —
# GR_pred = RMS_dB(pred) − RMS_dB(dry) — exactly the definition used for the target
# in eval_lstm_condlstm_gr_bins.ipynb, so this notebook reports the SAME GR plots
# and GR MAE numbers.

DEFAULT_PRE_ROLL_SEC = 10.0


def pair_paths(setting: str, song: str) -> tuple[str, str, str]:
    dry = os.path.join(DATA_ROOT, "processed_normalized", f"{song}_UnmasteredWAV.wav")
    wet = os.path.join(wet_dir_for_setting(DATA_ROOT, setting), f"{song}-exported.wav")
    gr = os.path.join(DATA_ROOT, "gr_curves", setting, f"{song}.pt")
    return dry, wet, gr


def _params_for(setting: str) -> torch.Tensor:
    # Normalised static knobs [1, num_controls] from the setting folder name.
    return torch.tensor(normalize_setting_params(setting), dtype=torch.float32,
                        device=DEVICE).unsqueeze(0)


def _load_gr_segment(gr_path: str, start: int, stop: int) -> torch.Tensor:
    gr_db = torch.load(gr_path, map_location="cpu", weights_only=False)["gr_db"].float()
    if gr_db.dim() == 1:
        gr_db = gr_db.unsqueeze(0)
    return gr_db[..., start:stop]


def _gr_from_audio(dry: torch.Tensor, sig: torch.Tensor) -> np.ndarray:
    # Gain reduction (dB) of `sig` relative to `dry`, clamped to the GR range.
    return (
        gain_reduction_db(dry, sig, RMS_WIN).clamp(GR_DB_MIN, GR_DB_MAX).squeeze().cpu().numpy()
    )


@torch.no_grad()
def predict_wet_segment(setting, song, start_sec=0.0, duration_sec=6.0,
                        pre_roll_sec=DEFAULT_PRE_ROLL_SEC, model=None):
    model = model or MODEL
    sr = SR
    dry_path, wet_path, gr_path = pair_paths(setting, song)

    start = int(round(start_sec * sr))
    stop = start + int(round(duration_sec * sr))
    context_start = max(0, start - int(round(pre_roll_sec * sr)))
    offset = start - context_start   # warm-up length (no conv left-context anymore)

    dry_ctx, wet_ctx = _read_dry_wet_segment(dry_path, wet_path, context_start, stop, sr)
    gr_ctx = _load_gr_segment(gr_path, context_start, stop)
    n = min(dry_ctx.shape[-1], gr_ctx.shape[-1], wet_ctx.shape[-1])
    dry_ctx, gr_ctx, wet_ctx = dry_ctx[..., :n], gr_ctx[..., :n], wet_ctx[..., :n]
    params = _params_for(setting)

    model.reset_states()
    pred_full = model(dry_ctx.unsqueeze(0).to(DEVICE), gr_ctx.unsqueeze(0).to(DEVICE), params)

    seg_len = min(stop - start, pred_full.shape[-1] - offset)
    sl = slice(offset, offset + seg_len)
    pred = pred_full[..., sl].squeeze(0).cpu()
    wet = wet_ctx[..., sl].cpu()
    dry = dry_ctx[..., sl].cpu()
    err = (pred - wet).squeeze().numpy()

    # Recover GR curves from the waveforms (same definition as the GR-bins model).
    gr_true = _gr_from_audio(dry, wet)
    gr_pred = _gr_from_audio(dry, pred)
    gr_err = gr_pred - gr_true

    t_len = pred.shape[-1]
    return {
        "song": song, "setting": setting,
        "sample_rate": sr, "start_sec": start_sec, "duration_sec": t_len / sr,
        "pre_roll_sec": offset / sr, "time": np.arange(t_len) / sr,
        "dry": dry.squeeze().numpy(),
        "wet": wet.squeeze().numpy(), "pred": pred.squeeze().numpy(),
        "error": err,
        "gr_true": gr_true, "gr_pred": gr_pred, "gr_error": gr_err,
        "mae": float(np.mean(np.abs(err))),
        "esr": float(esr_metric(wet.unsqueeze(0), pred.unsqueeze(0))),
        "gr_mae_db": float(np.mean(np.abs(gr_err))),
        "gr_rmse_db": float(np.sqrt(np.mean(gr_err ** 2))),
    }


def visualize_prediction(setting=None, song=None, start_sec=30.0, duration_sec=6.0,
                         pre_roll_sec=DEFAULT_PRE_ROLL_SEC, title=None):
    song = song or TEST_PAIRS[0][0]
    setting = setting or TEST_PAIRS[0][1]
    seg = predict_wet_segment(setting, song, start_sec, duration_sec, pre_roll_sec)

    t = seg["time"]
    # rows: dry | waveform (wet/pred) | waveform error | GR (true/pred) | GR error
    fig, axes = plt.subplots(5, 1, figsize=(11, 13), sharex=True,
                             gridspec_kw={"height_ratios": [1, 2, 1, 2, 1]})

    axes[0].plot(t, seg["dry"], lw=0.6, color="#444")
    axes[0].set_ylabel("dry"); axes[0].set_ylim(-1.05, 1.05); axes[0].grid(alpha=0.3)

    axes[1].plot(t, seg["wet"], label="target (wet)", lw=1.0, color="#1f77b4")
    axes[1].plot(t, seg["pred"], label="prediction", lw=0.9, color="#d62728", alpha=0.85)
    axes[1].set_ylabel("amp"); axes[1].legend(loc="lower right"); axes[1].set_ylim(-1.05, 1.05)
    axes[1].grid(alpha=0.3)

    axes[2].plot(t, seg["error"], lw=0.8, color="#2ca02c")
    axes[2].axhline(0, color="k", lw=0.5, alpha=0.4)
    axes[2].set_ylabel("wave err"); axes[2].grid(alpha=0.3)

    axes[3].plot(t, seg["gr_true"], label="target", lw=1.4, color="#1f77b4")
    axes[3].plot(t, seg["gr_pred"], label="prediction", lw=1.2, color="#d62728", alpha=0.85)
    axes[3].axhline(0, color="k", lw=0.5, alpha=0.4)
    axes[3].set_ylabel("GR (dB)"); axes[3].legend(loc="lower right"); axes[3].grid(alpha=0.3)

    axes[4].plot(t, seg["gr_error"], lw=0.8, color="#9467bd")
    axes[4].axhline(0, color="k", lw=0.5, alpha=0.4)
    axes[4].set_ylabel("GR err (dB)"); axes[4].set_xlabel("time (s)"); axes[4].grid(alpha=0.3)

    fig.suptitle(title or (
        f"{seg['song']}  |  {seg['setting']}\n"
        f"pre-roll={seg['pre_roll_sec']:.1f}s  "
        f"MAE {seg['mae']:.4f}  ESR {seg['esr']:.4f}  "
        f"GR MAE={seg['gr_mae_db']:.2f} dB  GR RMSE={seg['gr_rmse_db']:.2f} dB"
    ), fontsize=10)
    plt.tight_layout(); plt.show()
    return seg


def settings_sweep(song=None, settings=None, start_sec=30.0, duration_sec=6.0,
                   pre_roll_sec=DEFAULT_PRE_ROLL_SEC):
    # Same song through every setting - generalisation across knobs.
    # Compares the gain-reduction curves (target vs prediction) per setting.
    song = song or SPLIT.val_songs[0]
    settings = settings or SPLIT.all_settings
    segs = [predict_wet_segment(s, song, start_sec, duration_sec, pre_roll_sec) for s in settings]

    fig, axes = plt.subplots(len(segs), 1, figsize=(11, 1.8 * len(segs)),
                             sharex=True, sharey=True, squeeze=False)
    for ax, s in zip(axes[:, 0], segs):
        ax.plot(s["time"], s["gr_true"], lw=1.4, color="#1f77b4", label="target")
        ax.plot(s["time"], s["gr_pred"], lw=1.2, color="#d62728", alpha=0.85, label="prediction")
        ax.axhline(0, color="k", lw=0.5, alpha=0.4)
        ax.set_ylabel("GR (dB)", fontsize=8)
        ax.set_title(
            f"{s['setting']}  —  GR MAE {s['gr_mae_db']:.2f} dB  "
            f"GR RMSE {s['gr_rmse_db']:.2f} dB  MAE {s['mae']:.4f}  ESR {s['esr']:.4f}",
            fontsize=8, loc="left",
        )
        ax.grid(alpha=0.3)
    axes[0, 0].legend(loc="lower right", fontsize=8)
    axes[-1, 0].set_xlabel("time (s)")
    fig.suptitle(f"Settings sweep — GR curves — {song} ({duration_sec:.0f}s @ {start_sec:.0f}s)",
                 fontsize=11)
    plt.tight_layout(); plt.show()

    df = pd.DataFrame([
        {
            "setting": s["setting"],
            "GR MAE (dB)": s["gr_mae_db"],
            "MAE": s["mae"],
            "ESR": s["esr"],
        }
        for s in segs
    ])
    display(df)
    return segs, df

## Usage

Re-run the cells below any time; they read the latest synced files from Drive.
Use the available-runs cell first, then set `SELECTED_RUN_NAME` in the selection cell.

- `visualize_prediction(setting=..., song=...)` — any (setting, song); defaults to the first **test** pair. Shows the waveform (target / prediction) **and** the gain-reduction curve recovered from the prediction (`GR = RMS_dB(out) − RMS_dB(dry)`), mirroring the GR plots in [`eval_lstm_condlstm_gr_bins.ipynb`](../05_conditioning/eval_lstm_condlstm_gr_bins.ipynb).
- `settings_sweep(song=...)` — one song through all 10 settings (defaults to the val song). Plots the **gain-reduction curve comparison** (target vs prediction) per setting; the per-setting `GR MAE / MAE / ESR` table is the headline generalisation result.
- The split metrics cell scores the **validation** split on the canonical 9-column engine (`GR MAE (dB)`, `MR-STE`, `MR-STFT`, `ESR (A-wt)`, `MAE (L1)`, `MSE (L2)`, `EDC`, `M_NRMSE`, `M_SF`) ported from [`02b_sota_training/eval_lstm_diffssl_tvc.ipynb`](../02b_sota_training/eval_lstm_diffssl_tvc.ipynb), so numbers are directly comparable across models.
- The final section scores the **held-out test set** (`test_ground_truth/`, 5 fully-unseen song×setting pairs) with the same engine: cell 8a exports the missing GR curves to `gr_curves/<setting>/<song>.pt`, then 8b loads them and streams exactly like the validation cell. (The `SPLIT.test_pair_keys` split over `processed_ground_truth` is still intentionally skipped.)

In [8]:
# Single prediction preview — held-out test song x lowest-threshold setting
song, setting = TEST_PAIRS[0]
_ = visualize_prediction(setting=setting, song=song, start_sec=30.0, duration_sec=6.0)


<Figure size 1100x1300 with 5 Axes>

In [9]:
# Settings sweep: validation song through all 10 settings
sweep_segs, sweep_df = settings_sweep(start_sec=30.0, duration_sec=6.0)


<Figure size 1100x1800 with 10 Axes>

,setting,GR MAE (dB),MAE,ESR
0,threshold_-12_attack_10_release_0.4_ratio_10,0.195393,0.000379,0.006435
1,threshold_-12_attack_1_release_0.1_ratio_2,0.183635,0.000399,0.006860
2,threshold_-4_attack_10_release_0.1_ratio_2,0.093015,0.000715,0.013847
3,threshold_-4_attack_1_release_0.4_ratio_10,0.225382,0.000803,0.013234
4,threshold_-8_attack_30_release_0.8_ratio_4,0.072900,0.000494,0.009533
5,threshold_0_attack_3_release_0.8_ratio_4,0.101527,0.000752,0.014743
6,threshold_12_attack_3_release_0.8_ratio_2,0.118854,0.001068,0.021452
7,threshold_4_attack_10_release_0.1_ratio_10,0.139696,0.001126,0.022168
8,threshold_8_attack_1_release_0.1_ratio_10,0.100788,0.001087,0.022038
9,threshold_8_attack_30_release_0.4_ratio_2,0.108290,0.001019,0.020649


In [10]:
# ── 6b. Metric engine (ported from 02b_sota_training/eval_lstm_diffssl_tvc.ipynb) ──
#
# The canonical 9-column table so this model drops straight into the cross-model
# comparison. Every moving-average envelope uses an O(N) cumsum window; the FFT
# metrics reimplement auraloss MultiResolutionSTFTLoss and src.losses.{M_SF, EDC},
# batched. MR-STFT and ESR are verified below against nablafx.evaluation (auraloss)
# to <1e-3 / <1e-4.

from scipy.signal import bilinear, lfilter

EPS = 1e-10

# Multi-resolution window / FFT sets (identical to the diffssl_tvc engine)
MR_STE_WINDOWS   = (256, 1024, 4096)                     # short-time energy (envelope shape)
MR_NRMSE_WINDOWS = (512, 1024, 2048)                     # == src.losses.multi_resolution_nrmse
STFT_SIZES       = (512, 1024, 2048)                     # spectral-flux resolutions
_MRSTFT_RES      = [(1024, 120, 600), (2048, 240, 1200), (512, 50, 240)]   # auraloss defaults

# Final metric columns: GR-stage | colour-stage | standard
METRIC_COLS = [
    "GR MAE (dB)", "MR-STE",                             # GR stage
    "MR-STFT", "ESR (A-wt)",                             # colour stage (auraloss MR-STFT + A-weighted ESR)
    "MAE (L1)", "MSE (L2)", "EDC", "M_NRMSE", "M_SF",    # standard
]


# ── moving-average envelopes (cumsum: O(N), window-independent) ──
def _moving_meansq(x_sq, W, centered):
    """Sliding mean of x**2 via prefix sums. centered=symmetric, else causal (trailing)."""
    n = x_sq.shape[-1]
    c = np.empty(n + 1); c[0] = 0.0; np.cumsum(x_sq, out=c[1:])
    i = np.arange(n)
    if centered:
        lo = i - (W // 2); hi = lo + W
    else:                                                # causal == src.dsp_torch.gain_reduction_db
        hi = i + 1; lo = hi - W
    lo = np.clip(lo, 0, n); hi = np.clip(hi, 0, n)
    return (c[hi] - c[lo]) / np.maximum(hi - lo, 1)

def _to_db(x):       return 20.0 * np.log10(np.maximum(x, EPS))
def _esr(pred, tgt): return float(np.sum((tgt - pred) ** 2) / (np.sum(tgt * tgt) + 1e-8))

# A-weighting IIR (analog prototype -> bilinear) for the A-weighted ESR
def _a_weighting_ba(fs):
    f1, f2, f3, f4 = 20.598997, 107.65265, 737.86223, 12194.217
    A1000 = 1.9997
    nums = [(2 * np.pi * f4) ** 2 * 10 ** (A1000 / 20.0), 0, 0, 0, 0]
    dens = np.polymul([1, 4 * np.pi * f4, (2 * np.pi * f4) ** 2],
                      [1, 4 * np.pi * f1, (2 * np.pi * f1) ** 2])
    dens = np.polymul(np.polymul(dens, [1, 2 * np.pi * f3]), [1, 2 * np.pi * f2])
    return bilinear(nums, dens, fs)
AW_B, AW_A = _a_weighting_ba(SR)


def reference_gr_db(dry, wet, W=RMS_WIN):
    """Reference GR trajectory = causal RMS GR (matches src.dsp_torch.gain_reduction_db)."""
    return (_to_db(np.sqrt(_moving_meansq(wet * wet, W, centered=False)))
            - _to_db(np.sqrt(_moving_meansq(dry * dry, W, centered=False))))


# ── GR-stage / envelope metrics ──
def mr_ste(pred, wet, windows=MR_STE_WINDOWS):
    """Multi-resolution short-time energy distance (envelope shape; Wright & Valimaki)."""
    pe, we = pred * pred, wet * wet
    total = 0.0
    for W in windows:
        ep = _moving_meansq(pe, W, centered=True)
        et = _moving_meansq(we, W, centered=True)
        total += np.sum(np.abs(ep - et)) / (np.sum(np.abs(et)) + 1e-8)
    return float(total / len(windows))

def mr_nrmse(pred, wet, windows=MR_NRMSE_WINDOWS):
    """Mean normalised RMS-envelope error (== src.losses.multi_resolution_nrmse, cumsum-fast)."""
    total = 0.0
    for W in windows:
        rp = np.sqrt(_moving_meansq(pred * pred, W, centered=False))
        rt = np.sqrt(_moving_meansq(wet * wet, W, centered=False))
        total += np.sqrt(np.mean((rt - rp) ** 2)) / (np.sqrt(np.mean(rt * rt)) + 1e-8)
    return float(total / len(windows))


# ── colour-stage + standard FFT metrics ──
_HANN = {}
def _win(n):
    if n not in _HANN:
        _HANN[n] = torch.hann_window(n)
    return _HANN[n]

def _stft_mag_pow(x2d, n_fft, hop, win):
    """auraloss-style magnitude = sqrt(clamp(re^2 + im^2, 1e-8))."""
    st = torch.stft(x2d, n_fft=n_fft, hop_length=hop, win_length=win,
                    window=_win(win), return_complex=True)
    return torch.sqrt(torch.clamp(st.real ** 2 + st.imag ** 2, min=1e-8))

def _stft_mag_fft(x2d, n_fft):
    """src.losses-style magnitude (hop = n_fft // 4, hann(n_fft))."""
    return torch.stft(x2d, n_fft=n_fft, hop_length=n_fft // 4,
                      window=_win(n_fft), return_complex=True).abs()

@torch.no_grad()
def fft_metrics(pred1d: torch.Tensor, wet1d: torch.Tensor) -> dict:
    """auraloss MR-STFT + src.losses {M_SF, EDC} for one (pred, wet) pair."""
    P1, T1 = pred1d.unsqueeze(0), wet1d.unsqueeze(0)
    out = {}

    # MR-STFT (auraloss default = spectral-convergence + log-magnitude, mean over 3 resolutions)
    acc = 0.0
    for n_fft, hop, win in _MRSTFT_RES:
        P = _stft_mag_pow(P1, n_fft, hop, win)[0]
        T = _stft_mag_pow(T1, n_fft, hop, win)[0]
        sc = torch.linalg.norm(T - P) / torch.linalg.norm(T)
        lm = (torch.log(P) - torch.log(T)).abs().mean()
        acc += float(sc + lm)
    out["MR-STFT"] = acc / len(_MRSTFT_RES)

    # spectral-flux (M_SF) over (512, 1024, 2048)
    sf = 0.0
    for n_fft in STFT_SIZES:
        P = _stft_mag_fft(P1, n_fft)[0]
        T = _stft_mag_fft(T1, n_fft)[0]
        pf, tf = torch.diff(P, dim=-1), torch.diff(T, dim=-1)
        sf += float((tf - pf).abs().mean() / (tf.abs().mean() + 1e-8))
    out["M_SF"] = sf / len(STFT_SIZES)

    # EDC (energy-decay-curve error, dB)
    pe = torch.flip(torch.cumsum(torch.flip(P1 ** 2, dims=(-1,)), dim=-1), dims=(-1,))
    te = torch.flip(torch.cumsum(torch.flip(T1 ** 2, dims=(-1,)), dim=-1), dims=(-1,))
    pe = pe / (pe[..., :1] + 1e-8); te = te / (te[..., :1] + 1e-8)
    out["EDC"] = float((10 * torch.log10(te.clamp(min=1e-8))
                        - 10 * torch.log10(pe.clamp(min=1e-8))).abs().mean())
    return out


def chunk_metrics(dry: np.ndarray, pred: np.ndarray, wet: np.ndarray) -> dict:
    """The 9 metric columns for one (dry, pred, wet) chunk (float64 1-D arrays).

    GR MAE compares the prediction's causal-RMS GR against the reference RMS
    GR(dry, wet), both clamped to [GR_DB_MIN, GR_DB_MAX]. FFT metrics run in float32.
    """
    gr_tgt  = np.clip(reference_gr_db(dry, wet),  GR_DB_MIN, GR_DB_MAX)
    gr_pred = np.clip(reference_gr_db(dry, pred), GR_DB_MIN, GR_DB_MAX)
    row = {
        "GR MAE (dB)": float(np.mean(np.abs(gr_pred - gr_tgt))),
        "MR-STE":      mr_ste(pred, wet),
        "ESR (A-wt)":  _esr(lfilter(AW_B, AW_A, pred), lfilter(AW_B, AW_A, wet)),
        "MAE (L1)":    float(np.mean(np.abs(pred - wet))),
        "MSE (L2)":    float(np.mean((pred - wet) ** 2)),
        "M_NRMSE":     mr_nrmse(pred, wet),
    }
    row.update(fft_metrics(torch.from_numpy(pred).float(),
                           torch.from_numpy(wet).float()))
    return row


# ── verify MR-STFT / ESR against nablafx.evaluation (auraloss) ──
try:
    from nablafx.evaluation import get_function
    _g = torch.Generator().manual_seed(0)
    _p = torch.randn(SR, generator=_g); _t = torch.randn(SR, generator=_g)
    _nab_mrstft, _nab_esr = get_function("mrstft_loss"), get_function("esr_loss")
    _pt, _tt = _p.view(1, 1, -1), _t.view(1, 1, -1)
    _fm = fft_metrics(_p, _t)
    _d_mrstft = abs(float(_nab_mrstft(_pt, _tt)) - _fm["MR-STFT"])
    _d_esr = abs(float(_nab_esr(_pt, _tt)) - _esr(_p.double().numpy(), _t.double().numpy()))
    assert _d_mrstft < 1e-3, _d_mrstft
    assert _d_esr < 1e-4, _d_esr
    print(f"nablafx.evaluation verified — MR-STFT Δ={_d_mrstft:.2e}, ESR Δ={_d_esr:.2e}")
except Exception as e:
    print(f"nablafx verification skipped: {e}")

print(f"Metric engine registered — {len(METRIC_COLS)} columns: {METRIC_COLS}")

nablafx.evaluation verified — MR-STFT Δ=0.00e+00, ESR Δ=5.61e-08
Metric engine registered — 9 columns: ['GR MAE (dB)', 'MR-STE', 'MR-STFT', 'ESR (A-wt)', 'MAE (L1)', 'MSE (L2)', 'EDC', 'M_NRMSE', 'M_SF']


In [11]:
# ── 7. Audio metrics on the VALIDATION split (TRUE stateful streaming) ──
# Streams each (song, setting) validation pair whole-song in NON-overlapping chunks
# with all internal LSTM states carried across chunks (matching TBPTT training):
# the main LSTM + the tvcond generator + the GR-TFiLM. reset_states() once per
# pair, detach_states() between chunks. The raw dry, the 4 static knobs, and the
# GR curve are fed for each chunk. Chunk length is snapped to a multiple of the
# tvcond/GR-TFiLM block (128). Each chunk is scored with the ported metric engine
# (chunk_metrics) — the SAME 9-column table as eval_lstm_diffssl_tvc.ipynb, results
# duration-weighted. (Test split intentionally skipped — will handle it later.)

MAX_EVAL_PAIRS: int | None = None    # small int for a smoke test
STREAM_CHUNK_SEC = 10.0
SAVE_CHUNK_METRICS = False
MIN_METRIC_FRAMES = max(STFT_SIZES)  # skip sub-FFT tail chunks (torch.stft needs >= n_fft)

sr = SR
_BLOCK = int(HP["model"].get("cond_block_size", 128))
# snap chunk length to a whole number of conditioning blocks
chunk_samples = max(_BLOCK, (int(round(STREAM_CHUNK_SEC * sr)) // _BLOCK) * _BLOCK)


@torch.no_grad()
def stream_pair_metrics(song: str, setting: str) -> list[dict]:
    # Stateful pass over a whole (song, setting) pair; per-chunk metric rows.
    dry_path, wet_path, gr_path = pair_paths(setting, song)
    total = _pair_num_frames(dry_path, wet_path, sr)
    gr_full = torch.load(gr_path, map_location="cpu", weights_only=False)["gr_db"].float()
    if gr_full.dim() == 1:
        gr_full = gr_full.unsqueeze(0)
    params = _params_for(setting)

    MODEL.reset_states()
    rows = []
    for ci, o in enumerate(range(0, total, chunk_samples), start=1):
        stop = min(o + chunk_samples, total)
        dry, wet = _read_dry_wet_segment(dry_path, wet_path, o, stop, sr)
        gr = gr_full[..., o:stop]
        n = min(dry.shape[-1], gr.shape[-1], wet.shape[-1])
        if n == 0:
            break
        dry, gr, wet = dry[..., :n], gr[..., :n], wet[..., :n]

        pred = MODEL(dry.unsqueeze(0).to(DEVICE), gr.unsqueeze(0).to(DEVICE), params)
        MODEL.detach_states()
        if n < MIN_METRIC_FRAMES:                       # too short for the largest STFT
            continue

        # float64 1-D arrays for the ported metric engine (FFT metrics run in float32).
        d = dry.squeeze(0).double().cpu().numpy()
        w = wet.squeeze(0).double().cpu().numpy()
        p = pred.squeeze().double().cpu().numpy()
        rows.append({"Chunk": ci, "Start (s)": o / sr, "Duration (s)": n / sr, "Frames": n,
                     **chunk_metrics(d, p, w)})
    return rows


val_pairs = VAL_PAIRS if MAX_EVAL_PAIRS is None else VAL_PAIRS[:MAX_EVAL_PAIRS]
chunk_rows, pair_rows = [], []
for i, (song, setting) in enumerate(val_pairs, start=1):
    print(f"[validation {i}/{len(val_pairs)}] {song} / {setting}")
    rows = stream_pair_metrics(song, setting)
    chunk_rows += [{"Song": song, "Setting": setting, **r} for r in rows]
    pair_rows.append({
        "Split": "validation", "Song": song, "Setting": setting,
        "Duration (s)": sum(r["Frames"] for r in rows) / sr,
        **_weighted_average_metric_rows(rows, METRIC_COLS),
    })
    gc.collect()

total_frames = sum(r["Frames"] for r in chunk_rows)
split_row = {
    "Split": "validation", "Pairs": len(val_pairs), "Chunks": len(chunk_rows),
    "Duration (s)": total_frames / sr,
    **_weighted_average_metric_rows(chunk_rows, METRIC_COLS),
}

pair_metrics_df = pd.DataFrame(pair_rows)
audio_metrics_df = pd.DataFrame([split_row])
pair_metrics_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_validation_pairs.csv", index=False)
audio_metrics_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_validation.csv", index=False)
if SAVE_CHUNK_METRICS:
    pd.DataFrame(chunk_rows).to_csv(
        Path(RUN_DIR) / "eval_audio_metrics_validation_chunks.csv", index=False)
print(f"Saved -> {RUN_DIR}/eval_audio_metrics_validation(.csv, _pairs.csv)")

display(pair_metrics_df)
display(audio_metrics_df)

[validation 1/10] Ecstasy / threshold_-12_attack_10_release_0.4_ratio_10
[validation 2/10] Ecstasy / threshold_-12_attack_1_release_0.1_ratio_2
[validation 3/10] Ecstasy / threshold_-4_attack_10_release_0.1_ratio_2
[validation 4/10] Ecstasy / threshold_-4_attack_1_release_0.4_ratio_10
[validation 5/10] Ecstasy / threshold_-8_attack_30_release_0.8_ratio_4
[validation 6/10] Ecstasy / threshold_0_attack_3_release_0.8_ratio_4
[validation 7/10] Ecstasy / threshold_12_attack_3_release_0.8_ratio_2
[validation 8/10] Ecstasy / threshold_4_attack_10_release_0.1_ratio_10
[validation 9/10] Ecstasy / threshold_8_attack_1_release_0.1_ratio_10
[validation 10/10] Ecstasy / threshold_8_attack_30_release_0.4_ratio_2
Saved -> /Volumes/Saola's Drive/AllCode/thesis/data/diffssl_gr_tfilm_runs/gr_tfilm_20260701_195700_diffssl_lstm32_tvc_gr_tfilm/eval_audio_metrics_validation(.csv, _pairs.csv)


,Split,Song,Setting,Duration (s),GR MAE (dB),MR-STE,MR-STFT,ESR (A-wt),MAE (L1),MSE (L2),EDC,M_NRMSE,M_SF
0,validation,Ecstasy,threshold_-12_attack_10_release_0.4_ratio_10,257.746463,0.377754,0.057931,0.102493,0.001639,0.000324,1.845035e-07,0.213646,0.040199,0.068260
1,validation,Ecstasy,threshold_-12_attack_1_release_0.1_ratio_2,257.746463,0.438647,0.127317,0.131863,0.001806,0.000360,2.179266e-07,0.291934,0.064555,0.069213
2,validation,Ecstasy,threshold_-4_attack_10_release_0.1_ratio_2,257.746463,0.306678,0.061134,0.090646,0.000358,0.000605,6.512685e-07,0.251795,0.036711,0.038748
3,validation,Ecstasy,threshold_-4_attack_1_release_0.4_ratio_10,257.746463,0.431839,0.247055,0.183698,0.001933,0.000647,8.032719e-07,0.325476,0.092626,0.077617
4,validation,Ecstasy,threshold_-8_attack_30_release_0.8_ratio_4,257.746463,0.308313,0.123661,0.112430,0.000321,0.000412,2.907920e-07,0.294121,0.054760,0.036929
5,validation,Ecstasy,threshold_0_attack_3_release_0.8_ratio_4,257.746463,0.338854,0.198985,0.137320,0.000389,0.000619,6.789842e-07,0.307939,0.074075,0.039444
6,validation,Ecstasy,threshold_12_attack_3_release_0.8_ratio_2,257.746463,0.304305,0.045890,0.073442,0.000306,0.000916,1.533464e-06,0.185915,0.028647,0.031069
7,validation,Ecstasy,threshold_4_attack_10_release_0.1_ratio_10,257.746463,0.358287,0.087624,0.099519,0.000371,0.000982,1.801998e-06,0.275922,0.048418,0.033412
8,validation,Ecstasy,threshold_8_attack_1_release_0.1_ratio_10,257.746463,0.290774,0.046302,0.075033,0.000247,0.000937,1.637715e-06,0.228026,0.029741,0.028733
9,validation,Ecstasy,threshold_8_attack_30_release_0.4_ratio_2,257.746463,0.200209,0.035563,0.061151,0.000283,0.000883,1.434020e-06,0.087923,0.020795,0.030476


,Split,Pairs,Chunks,Duration (s),GR MAE (dB),MR-STE,MR-STFT,ESR (A-wt),MAE (L1),MSE (L2),EDC,M_NRMSE,M_SF
0,validation,10,260,2577.464626,0.335566,0.103146,0.106759,0.000765,0.000668,9.233943e-07,0.24627,0.049053,0.04539


## Held-out TEST set (`test_ground_truth`)

Final generalisation check on the **external held-out test set** in
`Diff-SSL-G-Comp/test_ground_truth/` — 5 `(song, setting)` pairs whose **songs
*and* compressor settings were never seen during training** (e.g.
`IncidenteEnIntag @ threshold_-12_attack_1_release_0.8_ratio_2`). This is
separate from `SPLIT.test_pair_keys` (those held-out `processed_ground_truth`
pairs remain unscored); it is the harder, fully-unseen test.

Two cells:

- **8a — export GR curves.** These unseen settings have no `gr_curves/*.pt`, so
  build them once with the canonical exporter
  (`03_initial_GR_pred/gr_dataset.py::export_pair`) — the exact same code and
  format (raw dB, 1024-sample causal RMS) as the training/validation curves,
  written to `gr_curves/<setting>/<song>.pt`. Files that already exist are
  skipped (set `FORCE_REEXPORT=True` to overwrite).
- **8b — metrics.** Scores the test set with the **exact same** stateful
  whole-song streaming and canonical 9-column engine as the validation cell
  (`GR MAE (dB)`, `MR-STE`, `MR-STFT`, `ESR (A-wt)`, `MAE (L1)`, `MSE (L2)`,
  `EDC`, `M_NRMSE`, `M_SF`) — loading the exported `.pt` and slicing it exactly
  like the validation path. Results are written to
  `eval_audio_metrics_test(.csv, _pairs.csv)` alongside the validation CSVs.

In [12]:
# ── 8a. Export GR curves for the held-out test settings ──────────────
# The unseen test settings have no precomputed gr_curves/*.pt, so build them ONCE
# with the canonical exporter (03_initial_GR_pred/gr_dataset.py::export_pair) —
# the exact same code, method and format (raw dB, 1024-sample causal RMS, chunked
# memory-safe read) used for the training/validation curves. After this the test
# eval below loads gr_curves/<setting>/<song>.pt and slices it, byte-for-byte
# identical to the validation path.

from gr_dataset import export_pair

TEST_GT_DIR = os.path.join(DATA_ROOT, "test_ground_truth")
FORCE_REEXPORT = False   # set True to overwrite existing test .pt files


def discover_test_pairs(test_gt_dir: str = TEST_GT_DIR) -> list[tuple[str, str]]:
    # (song, setting) pairs under test_ground_truth that have a dry match.
    dry_dir = os.path.join(DATA_ROOT, "processed_normalized")
    pairs, missing = [], []
    for setting in sorted(os.listdir(test_gt_dir)):
        sdir = os.path.join(test_gt_dir, setting)
        if not os.path.isdir(sdir) or setting.startswith("."):
            continue
        for wet in sorted(glob.glob(os.path.join(sdir, "*-exported.wav"))):
            song = os.path.basename(wet).replace("-exported.wav", "")
            dry = os.path.join(dry_dir, f"{song}_UnmasteredWAV.wav")
            (pairs if os.path.isfile(dry) else missing).append((song, setting))
    if missing:
        print(f"[warning] {len(missing)} test wet files have no dry match (skipped): {missing}")
    return pairs


def test_pair_paths(setting: str, song: str) -> tuple[str, str, str]:
    dry = os.path.join(DATA_ROOT, "processed_normalized", f"{song}_UnmasteredWAV.wav")
    wet = os.path.join(TEST_GT_DIR, setting, f"{song}-exported.wav")
    gr = os.path.join(DATA_ROOT, "gr_curves", setting, f"{song}.pt")
    return dry, wet, gr


NEW_TEST_PAIRS = discover_test_pairs()
print(f"Held-out test pairs (test_ground_truth): {len(NEW_TEST_PAIRS)}")

_wrote, _skipped = [], []
for song, setting in NEW_TEST_PAIRS:
    dry_path, wet_path, gr_path = test_pair_paths(setting, song)
    if os.path.isfile(gr_path) and not FORCE_REEXPORT:
        _skipped.append(gr_path)
        continue
    export_pair(
        dry_path, wet_path, gr_path,
        rms_window=RMS_WIN,
        metadata={"song": song, "setting": setting, "dataset": "diffssl"},
    )
    _wrote.append(gr_path)

print(f"GR curve export — wrote {len(_wrote)}, skipped {len(_skipped)} existing")
for p in _wrote:
    print(f"  wrote  {os.path.relpath(p, DATA_ROOT)}")
for p in _skipped:
    print(f"  exists {os.path.relpath(p, DATA_ROOT)}")

Held-out test pairs (test_ground_truth): 5
GR curve export — wrote 5, skipped 0 existing
  wrote  gr_curves/threshold_-12_attack_1_release_0.8_ratio_2/IncidenteEnIntag.pt
  wrote  gr_curves/threshold_-8_attack_3_release_0.4_ratio_4/OralHygiene.pt
  wrote  gr_curves/threshold_12_attack_1_release_0.1_ratio_2/54.pt
  wrote  gr_curves/threshold_4_attack_1_release_0.8_ratio_10/SuchFinePeople.pt
  wrote  gr_curves/threshold_4_attack_30_release_0.8_ratio_10/Convertible.pt


In [13]:
# ── 8b. Audio metrics on the held-out TEST set (test_ground_truth) ───
# Byte-for-byte the same as the VALIDATION streaming cell (── 7): stateful
# whole-song streaming in block-aligned chunks, all internal LSTM states carried
# across chunks (main LSTM + tvcond + GR-TFiLM), reset_states() once per pair,
# detach_states() between chunks, scored with the SAME 9-column engine and
# duration-weighted. Only the pair source differs: wet from test_ground_truth/
# and the GR curve from the .pt exported in 8a (songs AND settings unseen in
# training — true generalisation).

@torch.no_grad()
def stream_test_pair_metrics(song: str, setting: str) -> list[dict]:
    # Stateful pass over a whole (song, setting) TEST pair; per-chunk metric rows.
    dry_path, wet_path, gr_path = test_pair_paths(setting, song)
    total = _pair_num_frames(dry_path, wet_path, sr)
    gr_full = torch.load(gr_path, map_location="cpu", weights_only=False)["gr_db"].float()
    if gr_full.dim() == 1:
        gr_full = gr_full.unsqueeze(0)
    params = _params_for(setting)

    MODEL.reset_states()
    rows = []
    for ci, o in enumerate(range(0, total, chunk_samples), start=1):
        stop = min(o + chunk_samples, total)
        dry, wet = _read_dry_wet_segment(dry_path, wet_path, o, stop, sr)
        gr = gr_full[..., o:stop]
        n = min(dry.shape[-1], gr.shape[-1], wet.shape[-1])
        if n == 0:
            break
        dry, gr, wet = dry[..., :n], gr[..., :n], wet[..., :n]

        pred = MODEL(dry.unsqueeze(0).to(DEVICE), gr.unsqueeze(0).to(DEVICE), params)
        MODEL.detach_states()
        if n < MIN_METRIC_FRAMES:                       # too short for the largest STFT
            continue

        # float64 1-D arrays for the ported metric engine (FFT metrics run in float32).
        d = dry.squeeze(0).double().cpu().numpy()
        w = wet.squeeze(0).double().cpu().numpy()
        p = pred.squeeze().double().cpu().numpy()
        rows.append({"Chunk": ci, "Start (s)": o / sr, "Duration (s)": n / sr, "Frames": n,
                     **chunk_metrics(d, p, w)})
    return rows


test_chunk_rows, test_pair_rows = [], []
for i, (song, setting) in enumerate(NEW_TEST_PAIRS, start=1):
    print(f"[test {i}/{len(NEW_TEST_PAIRS)}] {song} / {setting}")
    rows = stream_test_pair_metrics(song, setting)
    test_chunk_rows += [{"Song": song, "Setting": setting, **r} for r in rows]
    test_pair_rows.append({
        "Split": "test", "Song": song, "Setting": setting,
        "Duration (s)": sum(r["Frames"] for r in rows) / sr,
        **_weighted_average_metric_rows(rows, METRIC_COLS),
    })
    gc.collect()

test_total_frames = sum(r["Frames"] for r in test_chunk_rows)
test_split_row = {
    "Split": "test", "Pairs": len(NEW_TEST_PAIRS), "Chunks": len(test_chunk_rows),
    "Duration (s)": test_total_frames / sr,
    **_weighted_average_metric_rows(test_chunk_rows, METRIC_COLS),
}

test_pair_metrics_df = pd.DataFrame(test_pair_rows)
test_audio_metrics_df = pd.DataFrame([test_split_row])
test_pair_metrics_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_test_pairs.csv", index=False)
test_audio_metrics_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_test.csv", index=False)
if SAVE_CHUNK_METRICS:
    pd.DataFrame(test_chunk_rows).to_csv(
        Path(RUN_DIR) / "eval_audio_metrics_test_chunks.csv", index=False)
print(f"Saved -> {RUN_DIR}/eval_audio_metrics_test(.csv, _pairs.csv)")

display(test_pair_metrics_df)
display(test_audio_metrics_df)

[test 1/5] IncidenteEnIntag / threshold_-12_attack_1_release_0.8_ratio_2
[test 2/5] OralHygiene / threshold_-8_attack_3_release_0.4_ratio_4
[test 3/5] 54 / threshold_12_attack_1_release_0.1_ratio_2
[test 4/5] SuchFinePeople / threshold_4_attack_1_release_0.8_ratio_10
[test 5/5] Convertible / threshold_4_attack_30_release_0.8_ratio_10
Saved -> /Volumes/Saola's Drive/AllCode/thesis/data/diffssl_gr_tfilm_runs/gr_tfilm_20260701_195700_diffssl_lstm32_tvc_gr_tfilm/eval_audio_metrics_test(.csv, _pairs.csv)


,Split,Song,Setting,Duration (s),GR MAE (dB),MR-STE,MR-STFT,ESR (A-wt),MAE (L1),MSE (L2),EDC,M_NRMSE,M_SF
0,test,IncidenteEnIntag,threshold_-12_attack_1_release_0.8_ratio_2,183.200000,0.245946,0.036020,0.066575,0.000658,0.000199,7.446586e-08,0.131952,0.022770,0.048882
1,test,OralHygiene,threshold_-8_attack_3_release_0.4_ratio_4,203.500000,0.313758,0.086480,0.159688,0.003612,0.000541,9.121313e-07,0.066713,0.057619,0.073132
2,test,54,threshold_12_attack_1_release_0.1_ratio_2,248.563651,0.234900,0.017276,0.047786,0.000144,0.000462,4.877752e-07,0.626458,0.011505,0.021870
3,test,SuchFinePeople,threshold_4_attack_1_release_0.8_ratio_10,220.243923,0.510614,7.277795,0.141848,0.007700,0.000908,1.758745e-06,0.479284,0.149877,0.054678
4,test,Convertible,threshold_4_attack_30_release_0.8_ratio_10,204.919025,0.331129,0.067287,0.123540,0.001483,0.000832,1.616473e-06,0.263391,0.043903,0.057284


,Split,Pairs,Chunks,Duration (s),GR MAE (dB),MR-STE,MR-STFT,ESR (A-wt),MAE (L1),MSE (L2),EDC,M_NRMSE,M_SF
0,test,5,109,1060.426599,0.327801,1.551423,0.106682,0.002726,0.000596,9.798916e-07,0.332882,0.057301,0.050032


## 9. Conditioning ablations — what does the GR input actually buy? (no training)

Everything above conditions the model on the **oracle** GR (computed from the
wet target — unavailable at deployment). Two no-training ablations pin down
what that means:

| Variant | GR source | What it measures |
|---|---|---|
| `oracle` | `gr_curves/*.pt` (from wet) | upper bound (the numbers above) |
| `predicted` | `05_conditioning` predictor on dry + knobs | **the deployable number** — the honest comparison against the no-GR SOTA (val MR-STFT 0.133) |
| `const_mean` | per-pair mean of the oracle GR | static gain with the right average — isolates the value of the GR's *temporal* information |
| `const_0db` | 0 dB everywhere | no GR information at all — the full contribution of the GR-TFiLM path |

Reading the table: `oracle → predicted` is the **deployment gap** (predictor
quality is the bottleneck if this is large); `oracle → const_mean` is the
**temporal-information contribution** of the conditioning mechanism (if these
are close, the TFiLM is barely using the curve's dynamics — consistent with
the val GR MAE being no better than the unconditioned SOTA); `const_0db` is
the sanity floor. All variants stream the SAME checkpoint with the SAME
9-column engine on the validation split.

In [ ]:
# ── 9a. Load the 05_conditioning GR predictor (deployable GR source) ──
# StatefulCondLSTMGRBins from gr_pred_runs (dry + 4 knobs -> 71-bin GR logits
# at frame rate, hop 256), streamed statefully over whole songs and decoded /
# upsampled to a sample-rate GR curve in dB with model.to_db().

import importlib.util

GR_PRED_RUNS = "/Volumes/Saola's Drive/AllCode/thesis/data/gr_pred_runs"
GR_PRED_RUN  = "lstm_gr_20260611_132730_lstm_condlstm_bins_lds_cdrop_coldstart"

# 05_conditioning module names collide with 06_output (model.py, splits.py, ...).
# Load its model.py by file path under a private name; append (not prepend) the
# dir so its bare `gr_target` import resolves without shadowing anything.
_COND05_DIR = str(REPO_ROOT / "05_conditioning")
if _COND05_DIR not in sys.path:
    sys.path.append(_COND05_DIR)

def _load_module_from(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

_model05 = _load_module_from(os.path.join(_COND05_DIR, "model.py"), "model05_gr_predictor")

with open(os.path.join(GR_PRED_RUNS, GR_PRED_RUN, "hparams.json")) as f:
    _gp_hp = json.load(f)
_gpm = _gp_hp["model"]
GR_PRED_MODEL = _model05.StatefulCondLSTMGRBins(
    hop_size=int(_gp_hp.get("hop_size", 256)),
    encoder_channels=int(_gpm["encoder_channels"]),
    hidden_size=int(_gpm["hidden_size"]),
    tfilm_channels=int(_gpm["tfilm_channels"]),
    tfilm_block_size=int(_gpm["tfilm_block_size"]),
    tfilm_num_layers=int(_gpm["tfilm_num_layers"]),
    num_bins=int(_gpm["num_bins"]),
    knob_freqs=int(_gpm["knob_freqs"]),
    knob_embed_dim=int(_gpm["knob_embed_dim"]),
    use_tfilm=bool(_gpm.get("use_tfilm", True)),
)
_gp_ckpt_dir = os.path.join(GR_PRED_RUNS, GR_PRED_RUN, "checkpoints")
_gp_bests = sorted(glob.glob(os.path.join(_gp_ckpt_dir, "best-*.ckpt")))
GR_PRED_CKPT = _gp_bests[-1] if _gp_bests else os.path.join(_gp_ckpt_dir, "last.ckpt")
_gp_sd = torch.load(GR_PRED_CKPT, map_location=DEVICE, weights_only=False)["state_dict"]
GR_PRED_MODEL.load_state_dict(
    {k[len("model."):]: v for k, v in _gp_sd.items() if k.startswith("model.")})
GR_PRED_MODEL.to(DEVICE).eval()
print(f"GR predictor: {sum(p.numel() for p in GR_PRED_MODEL.parameters()):,} params "
      f"({GR_PRED_RUN} / {os.path.basename(GR_PRED_CKPT)})")

GR_HOP = int(_gp_hp.get("hop_size", 256))
# chunk = multiple of hop x tfilm_block so the TFiLM blocks stay aligned across chunks
_GR_CHUNK_BLOCK = GR_HOP * int(_gpm["tfilm_block_size"])


@torch.no_grad()
def predict_gr_pair(dry_path, wet_path, params, total, chunk_sec=30.0):
    """Stateful whole-song GR prediction from dry + knobs -> [1, total] dB."""
    chunk = max(_GR_CHUNK_BLOCK,
                (int(round(chunk_sec * sr)) // _GR_CHUNK_BLOCK) * _GR_CHUNK_BLOCK)
    GR_PRED_MODEL.reset_cond_states()
    state, outs = None, []
    for o in range(0, total, chunk):
        stop = min(o + chunk, total)
        dry, _ = _read_dry_wet_segment(dry_path, wet_path, o, stop, sr)
        n = dry.shape[-1]
        if n < GR_HOP:                      # tail shorter than one frame
            if outs:
                outs.append(outs[-1][..., -1:].expand(1, n))
            break
        logits, state = GR_PRED_MODEL(dry.unsqueeze(0).to(DEVICE), params,
                                      state=state, return_state=True)
        outs.append(GR_PRED_MODEL.to_db(logits, sample_len=n).squeeze(0).cpu())
    gr = torch.cat(outs, dim=-1)
    if gr.shape[-1] < total:                # pad any sub-frame tail with the last value
        gr = torch.cat([gr, gr[..., -1:].expand(1, total - gr.shape[-1])], dim=-1)
    return gr[..., :total]


# ── quick sanity: predicted vs oracle GR on the first val pair ──
_song, _setting = VAL_PAIRS[0]
_dp, _wp, _gpth = pair_paths(_setting, _song)
_pv = _params_for(_setting)
_total_s = min(_pair_num_frames(_dp, _wp, sr), int(90 * sr))
_gr_hat = predict_gr_pair(_dp, _wp, _pv, _total_s)
_gr_ref = _load_gr_segment(_gpth, 0, _total_s)
_n = min(_gr_hat.shape[-1], _gr_ref.shape[-1])
_mae = float((_gr_hat[..., :_n].clamp(GR_DB_MIN, GR_DB_MAX)
              - _gr_ref[..., :_n].clamp(GR_DB_MIN, GR_DB_MAX)).abs().mean())
print(f"predictor GR MAE vs oracle — {_song} / {_setting} (first {_total_s/sr:.0f}s): {_mae:.3f} dB")

_a, _b = int(30 * sr), int(36 * sr)
_t = np.arange(_a, _b) / sr
plt.figure(figsize=(11, 3))
plt.plot(_t, _gr_ref[0, _a:_b].numpy(), lw=1.2, label="oracle GR (from wet)", color="#1f77b4")
plt.plot(_t, _gr_hat[0, _a:_b].numpy(), lw=1.0, label="predicted GR (dry + knobs)",
         color="#d62728", alpha=0.85)
plt.xlabel("time (s)"); plt.ylabel("GR (dB)"); plt.grid(alpha=0.3); plt.legend()
plt.title(f"{_song} | {_setting} — oracle vs predicted GR")
plt.tight_layout(); plt.show()

In [ ]:
# ── 9b. Run the GR-source ablation (validation split, same ckpt) ─────
# Streams the SAME GRTFiLMDiffSSLLSTM checkpoint with 4 different GR sources
# through the SAME 9-column metric engine. No training anywhere.

ABLATION_VARIANTS = ["oracle", "predicted", "const_mean", "const_0db"]
MAX_ABLATION_PAIRS: int | None = None    # small int (e.g. 3) for a smoke run


@torch.no_grad()
def stream_pair_metrics_with_gr(song: str, setting: str, gr_full: torch.Tensor) -> list[dict]:
    # stream_pair_metrics (cell 7) with an arbitrary, precomputed GR source.
    dry_path, wet_path, _ = pair_paths(setting, song)
    total = _pair_num_frames(dry_path, wet_path, sr)
    params = _params_for(setting)

    MODEL.reset_states()
    rows = []
    for ci, o in enumerate(range(0, total, chunk_samples), start=1):
        stop = min(o + chunk_samples, total)
        dry, wet = _read_dry_wet_segment(dry_path, wet_path, o, stop, sr)
        gr = gr_full[..., o:stop]
        n = min(dry.shape[-1], gr.shape[-1], wet.shape[-1])
        if n == 0:
            break
        dry, gr, wet = dry[..., :n], gr[..., :n], wet[..., :n]

        pred = MODEL(dry.unsqueeze(0).to(DEVICE), gr.unsqueeze(0).to(DEVICE), params)
        MODEL.detach_states()
        if n < MIN_METRIC_FRAMES:
            continue
        d = dry.squeeze(0).double().cpu().numpy()
        w = wet.squeeze(0).double().cpu().numpy()
        p = pred.squeeze().double().cpu().numpy()
        rows.append({"Chunk": ci, "Frames": n, **chunk_metrics(d, p, w)})
    return rows


abl_pairs = VAL_PAIRS if MAX_ABLATION_PAIRS is None else VAL_PAIRS[:MAX_ABLATION_PAIRS]
abl_chunk_rows = {v: [] for v in ABLATION_VARIANTS}
abl_pair_rows, _pred_gr_maes = [], []

for i, (song, setting) in enumerate(abl_pairs, start=1):
    dry_path, wet_path, gr_path = pair_paths(setting, song)
    total = _pair_num_frames(dry_path, wet_path, sr)
    gr_oracle = torch.load(gr_path, map_location="cpu", weights_only=False)["gr_db"].float()
    if gr_oracle.dim() == 1:
        gr_oracle = gr_oracle.unsqueeze(0)
    gr_oracle = gr_oracle[..., :total]

    sources = {}
    if "oracle" in ABLATION_VARIANTS:
        sources["oracle"] = gr_oracle
    if "predicted" in ABLATION_VARIANTS:
        gr_hat = predict_gr_pair(dry_path, wet_path, _params_for(setting), total)
        _n = min(gr_hat.shape[-1], gr_oracle.shape[-1])
        _pred_gr_maes.append(float(
            (gr_hat[..., :_n].clamp(GR_DB_MIN, GR_DB_MAX)
             - gr_oracle[..., :_n].clamp(GR_DB_MIN, GR_DB_MAX)).abs().mean()))
        sources["predicted"] = gr_hat
    if "const_mean" in ABLATION_VARIANTS:
        sources["const_mean"] = torch.full_like(gr_oracle, float(gr_oracle.mean()))
    if "const_0db" in ABLATION_VARIANTS:
        sources["const_0db"] = torch.zeros_like(gr_oracle)

    for variant in ABLATION_VARIANTS:
        print(f"[{i}/{len(abl_pairs)}] {song} / {setting} — {variant}")
        rows = stream_pair_metrics_with_gr(song, setting, sources[variant])
        abl_chunk_rows[variant] += rows
        abl_pair_rows.append({
            "Variant": variant, "Song": song, "Setting": setting,
            "Frames": sum(r["Frames"] for r in rows),
            **_weighted_average_metric_rows(rows, METRIC_COLS),
        })
    gc.collect()

if _pred_gr_maes:
    print(f"\npredictor GR MAE vs oracle over {len(_pred_gr_maes)} val pairs: "
          f"{np.mean(_pred_gr_maes):.3f} dB")

abl_split_df = pd.DataFrame([
    {"Variant": v, "Pairs": len(abl_pairs),
     "Duration (s)": sum(r["Frames"] for r in abl_chunk_rows[v]) / sr,
     **_weighted_average_metric_rows(abl_chunk_rows[v], METRIC_COLS)}
    for v in ABLATION_VARIANTS
])
abl_pair_df = pd.DataFrame(abl_pair_rows)

abl_pair_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_gr_ablation_pairs.csv", index=False)
abl_split_df.to_csv(Path(RUN_DIR) / "eval_audio_metrics_gr_ablation.csv", index=False)
print(f"Saved -> {RUN_DIR}/eval_audio_metrics_gr_ablation(.csv, _pairs.csv)")

# Reference rows for reading the table:
#   no-GR SOTA retrain (val):        MR-STFT 0.133 | GR MAE 0.314 dB
#   oracle amplitude match (no NN):  MR-STFT 0.069 | MR-STE 0.011 | GR MAE 0
display(abl_split_df)
display(abl_pair_df)